In [ ]:
import os
import pandas as pd
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
import astropy.units as u_astropy
from reproject import reproject_exact   # pip install reproject if missing

NATIVE_MAS = 20    # grizli mosaic 原始像素尺度 (mas/pixel)
TARGET_MAS  = 30   # 目标输出分辨率 (mas/pixel)
HALF_PIX    = 50   # 原始切图半径 (pixels @ 20mas，覆盖 1")

df = pd.read_csv("dja_low_res.csv")
#check all column names
print(df.columns)

In [ ]:
#unique file_phot, remove nan
root_original = df['file_phot'].unique()
#remove nan
root_original = [x for x in root_original if pd.notna(x)]

#loop through all root test
root = []
root_var = []
for index in range(len(root_original)):
    #skip nan
    if pd.isna(root_original[index]):
        continue
    else:
        root.append(root_original[index].replace('-fix_phot', '-f150w-clear_drc_sci')+'.gz')
print(root)
print(root_original)

In [ ]:
#try download all files in root to download_mosaic folder
from tqdm.auto import tqdm
os.makedirs('download_mosaic', exist_ok=True)

count = 0
for index in tqdm(range(len(root))):
    if os.path.exists('download_mosaic/'+root[index].replace('.gz', '')):
        continue
    try:
        #download without cell output
        os.system('wget -P download_mosaic/ https://s3.amazonaws.com/grizli-v2/JwstMosaics/v7/'+root[index])
        #then unzip the file
        os.system('gunzip download_mosaic/'+root[index])
        count += 1
    except Exception as e:
        continue

In [ ]:
from astropy.io import fits
from matplotlib import pyplot as plt
index =1
with fits.open('download_mosaic/'+root[index].replace('.gz', '')) as hdul:
    hdu = hdul[0]
    #print(hdu.header.get('PIXFRAC'))
    #print headers
    #print(hdu.header.keys)
    #unit in nanoJy
    #


In [ ]:
from astropy.io import fits
from matplotlib import pyplot as plt
from tqdm.auto import tqdm
from astropy.wcs import WCS
from astropy.table import Table
import numpy as np

scale  = TARGET_MAS / NATIVE_MAS          # 1.5
out_hw = int(HALF_PIX * 2 / scale)       # ~67 pixels at 30mas

# nJy/pixel → MJy/sr 换算系数（在原始 20mas 像素尺度下）
# 1 nJy = 1e-15 MJy；像素面积 = (NATIVE_MAS * 1e-3 arcsec)^2
_pix_arcsec2 = (NATIVE_MAS * 1e-3) ** 2                        # arcsec²
_pix_sr      = _pix_arcsec2 * (np.pi / (180.0 * 3600.0)) ** 2  # sr
NJY_TO_MJY_SR = 1e-15 / _pix_sr   # multiply cutout_nJy_pixel by this → MJy/sr
print(f'nJy/pixel → MJy/sr factor at {NATIVE_MAS}mas: {NJY_TO_MJY_SR:.4f}')

for image, name in tqdm(list(zip(root, root_original))[:20]):
    print('-------------------')
    print(image, name)
    try:
        with fits.open('download_mosaic/'+image.replace('.gz', '')) as hdul:
            hdu = hdul[0]
            catalog_sub = df[df['file_phot'] == name]
            print('Number of sources in catalog:', len(catalog_sub))
            wcs = WCS(hdu.header)
            index = np.random.randint(0, len(catalog_sub))
            print('Randomly selected source:', catalog_sub['objid'].values[index])
            ra  = catalog_sub['ra'].values[index]
            dec = catalog_sub['dec'].values[index]

            # --- 20mas cutout (nJy/pixel, with WCS) ---
            coord = SkyCoord(ra=ra*u_astropy.deg, dec=dec*u_astropy.deg)
            cutout_obj = Cutout2D(hdu.data, coord,
                                  size=(HALF_PIX*2, HALF_PIX*2),
                                  wcs=wcs, mode='partial', fill_value=0.0)
            cutout_20mas = cutout_obj.data          # nJy/pixel
            cutout_wcs   = cutout_obj.wcs
            print('20mas cutout shape:', cutout_20mas.shape)
            print('fraction of 0 in cutout:', np.sum(cutout_20mas == 0) / cutout_20mas.size)

            # --- Convert to MJy/sr before reprojecting ---
            # reproject_exact 对 MJy/sr（surface brightness）是严格正确的：
            # 面积加权平均保留表面亮度；而 nJy/pixel 是积分量，
            # 先转换可以避免面积比补偿，与 COSMOS-Web pipeline 完全统一。
            cutout_sb = cutout_20mas * NJY_TO_MJY_SR   # MJy/sr

            # --- Build output WCS: scaled pixel size, crpix/crval → source at center ---
            out_wcs = cutout_wcs.deepcopy()
            if out_wcs.wcs.has_cd():
                out_wcs.wcs.cd = out_wcs.wcs.cd * scale
            else:
                out_wcs.wcs.cdelt = out_wcs.wcs.cdelt * scale
            out_wcs.wcs.crpix = np.array([out_hw / 2 + 0.5, out_hw / 2 + 0.5])
            out_wcs.wcs.crval = np.array([ra, dec])
            out_wcs.wcs.set()

            # --- Reproject to 30mas (MJy/sr, flux-conserving for surface brightness) ---
            cutout_30mas, _ = reproject_exact(
                (cutout_sb, cutout_wcs),
                out_wcs,
                shape_out=(out_hw, out_hw)
            )   # output unit: MJy/sr, consistent with COSMOS-Web h5 files
            print('30mas cutout shape:', cutout_30mas.shape)
            print(f'value range (MJy/sr): [{np.nanmin(cutout_30mas):.4f}, {np.nanmax(cutout_30mas):.4f}]')

            # --- Plot comparison ---
            fig, axes = plt.subplots(1, 2, figsize=(8, 4))
            axes[0].imshow(cutout_20mas, origin='lower', cmap='gray')
            axes[0].set_title(f'20mas  nJy/pixel\n({cutout_20mas.shape[0]}×{cutout_20mas.shape[1]} px)')
            axes[1].imshow(cutout_30mas, origin='lower', cmap='gray')
            axes[1].set_title(f'30mas  MJy/sr\n({cutout_30mas.shape[0]}×{cutout_30mas.shape[1]} px)')
            fig.suptitle(name)
            plt.tight_layout()
            plt.show()

        # --- Spectrum ---
        table = fits.open(f'download/{catalog_sub["file"].values[index]}')[1].data
        wave = table['wave']
        mask = (wave > 1) & (wave < 2)
        wave = wave[mask]
        flux = table['flux'][mask]
        plt.plot(wave, flux)
        plt.xlim(1, 2)
        plt.title(name)
        plt.show()
    except Exception as e:
        print(e)

In [ ]:
!cd /raven/u/yacheng/projects/ssl_outthere/data/survey/DJA
!/u/yacheng/conda-envs/stenv/bin/python build_dja_image_h5.py \
    --num-workers 15 \
    --out dja_spectra_low_image.h5
